# 16 — Regional Zoom Selection & Hydrography

## Purpose

Notebook ini menjawab feedback:

> **“Di-zoom in lagi petanya. Jadi Kalimantannya spesifik di mana? Kalau di sampingnya ada sungai atau danau, kan tidak mungkin hotspot muncul di situ.”**

Tujuannya **belum membuat model baru**. Notebook ini memilih satu subregion Kalimantan secara data-driven, lalu memeriksa apakah resolusi spasial yang lebih halus lebih layak untuk tahap modeling berikutnya.

---

## Pertanyaan yang dijawab

1. Di bagian mana dari Kalimantan aktivitas FIRMS paling padat?
2. Subregion mana yang masih memiliki cukup banyak **3-day Emerging events** jika dibangun pada grid yang lebih halus?
3. Provinsi apa saja yang tercakup dalam kandidat subregion?
4. Apakah di dalam kandidat terdapat danau / sungai yang perlu diperhatikan?
5. Berapa banyak cell yang masih layak jika grid dibuat **0.25°** atau **0.10°**?
6. Apakah finer grid meningkatkan jumlah lokasi “Emerging” yang sebelumnya tersamarkan oleh grid 0.75°?

---

## Penting: zoom visual ≠ resolusi model

Memperbesar gambar 0.75° tidak menambah detail spasial.

Karena itu notebook ini melakukan dua hal sekaligus:

```text
Kalimantan-wide raw FIRMS
        ↓
data-driven regional selection
        ↓
regional hydrography inspection
        ↓
0.25° / 0.10° fine-grid preview
```

---

## Data selection principle

Subregion **tidak dipilih karena performa model**.

Kandidat dipilih dari raw FIRMS menggunakan:

1. jumlah 3-day Emerging positives pada preview grid 0.25°;
2. jumlah active days;
3. spatial spread / active fine-grid cells;
4. jumlah deteksi FIRMS.

Hydrography dipakai sebagai **descriptive / realism check**, bukan untuk memilih area yang “membuat model terlihat bagus”.

---

## Hydrography caveat

Notebook ini memakai **Natural Earth 1:10m** untuk screening visual:

- land;
- country / province boundaries;
- lakes;
- rivers + lake centerlines.

Layer ini bagus untuk regional context, tetapi **bukan exhaustive high-resolution water mask**. Sungai kecil, kanal, rawa, dan badan air kecil dapat tidak muncul.

Jika subregion sudah dipilih, Notebook 17 sebaiknya memakai water mask yang lebih detail (misalnya JRC Global Surface Water / HydroSHEDS-compatible source) untuk final fine-grid construction.

## 1. Environment

Jika package geospatial belum ada di `.venv`, jalankan di terminal:

```bash
python -m pip install geopandas shapely pyogrio requests folium
```

Lalu restart kernel.

In [61]:
# 1. Imports

from pathlib import Path
import json
import math
import re
import shutil
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import geopandas as gpd
    from shapely.geometry import Point, Polygon, box
    from shapely.ops import unary_union
except ImportError as exc:
    raise ImportError(
        "Notebook 16 requires geopandas + shapely. "
        "Install with: python -m pip install geopandas shapely pyogrio"
    ) from exc

try:
    import requests
except ImportError as exc:
    raise ImportError(
        "Install requests with: python -m pip install requests"
    ) from exc

try:
    import folium
    from folium.plugins import HeatMap
    FOLIUM_AVAILABLE = True
except ImportError:
    FOLIUM_AVAILABLE = False

warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 220)

print("geopandas:", gpd.__version__)
print("Folium available:", FOLIUM_AVAILABLE)

geopandas: 1.1.4
Folium available: True


## 2. Resolve project root and output folders

In [62]:
# 2. Project paths

cwd = Path.cwd()

candidate_roots = [
    cwd,
    cwd.parent,
    Path("/content/skripsi-karhut"),
    Path("/content/drive/MyDrive/skripsi-karhut"),
]

PROJECT_ROOT = None

for candidate in candidate_roots:
    corrected_nodes = (
        candidate
        / "data"
        / "intermediate"
        / "spatial_corrected"
        / "kalimantan_graph_nodes_corrected.csv"
    )

    if corrected_nodes.exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not detect project root from corrected spatial outputs."
    )

RESULTS_DIR = (
    PROJECT_ROOT
    / "model_results"
    / "regional_zoom_selection"
)

FIGURES_DIR = RESULTS_DIR / "figures"
EXTERNAL_DIR = PROJECT_ROOT / "data" / "external" / "natural_earth"
EXPORT_DIR = RESULTS_DIR / "exports"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("RESULTS_DIR  :", RESULTS_DIR)
print("EXTERNAL_DIR :", EXTERNAL_DIR)

PROJECT_ROOT : c:\Users\Asus\Documents\skripsi-karhut
RESULTS_DIR  : c:\Users\Asus\Documents\skripsi-karhut\model_results\regional_zoom_selection
EXTERNAL_DIR : c:\Users\Asus\Documents\skripsi-karhut\data\external\natural_earth


# PART A — Load raw FIRMS detections

## 3. Locate the original Kalimantan FIRMS daily folder

The packaged dataset documents daily files as:

```text
data/kalimantan/fire_firms/daily/fires_<date>.csv
```

Each row is one VIIRS-SNPP thermal-anomaly detection with `lat`, `lon`, and `frp`.

In [63]:
# 3. Discover raw FIRMS folder
# Prefer hourly because it is the more complete / authoritative FIRMS record.

from pathlib import Path

# ============================================================
# OPTIONAL MANUAL OVERRIDE
# ============================================================
# Kalau kamu tahu lokasi folder KARHUT_Data_Share, isi di sini.
# Contoh:
# DATASET_ROOT = Path(r"C:\Users\Asus\Documents\KARHUT_Data_Share")
#
# Kalau tidak tahu / ingin auto-detect, biarkan None.
# ============================================================

DATASET_ROOT = None


def has_firms_files(folder):
    return (
        folder.exists()
        and folder.is_dir()
        and any(folder.glob("fires_*.csv"))
    )


def find_firms_folder(dataset_root=None):
    """
    Search order:
    1. Manual DATASET_ROOT
    2. Inside skripsi-karhut
    3. Sibling folders of skripsi-karhut
    4. Common Documents / Downloads locations

    Prefer hourly over daily.
    """

    search_roots = []

    if dataset_root is not None:
        search_roots.append(Path(dataset_root))

    search_roots.extend([
        PROJECT_ROOT,
        PROJECT_ROOT.parent,
        Path.home() / "Documents",
        Path.home() / "Downloads",
    ])

    # Remove duplicate roots
    unique_roots = []
    seen = set()

    for root in search_roots:
        try:
            resolved = root.resolve()
        except Exception:
            resolved = root

        key = str(resolved).lower()

        if key not in seen and root.exists():
            unique_roots.append(root)
            seen.add(key)

    # --------------------------------------------------------
    # 1. Try common direct paths first
    # --------------------------------------------------------
    direct_candidates = []

    for root in unique_roots:

        direct_candidates.extend([
            # Dataset root itself
            root
            / "data"
            / "kalimantan"
            / "fire_firms"
            / "hourly",

            root
            / "data"
            / "kalimantan"
            / "fire_firms"
            / "daily",

            # KARHUT_Data_Share nested inside root
            root
            / "KARHUT_Data_Share"
            / "data"
            / "kalimantan"
            / "fire_firms"
            / "hourly",

            root
            / "KARHUT_Data_Share"
            / "data"
            / "kalimantan"
            / "fire_firms"
            / "daily",
        ])

    # Hourly first
    for candidate in direct_candidates:

        if (
            candidate.name.lower() == "hourly"
            and has_firms_files(candidate)
        ):
            return candidate, "hourly"

    # Daily fallback
    for candidate in direct_candidates:

        if (
            candidate.name.lower() == "daily"
            and has_firms_files(candidate)
        ):
            return candidate, "daily"

    # --------------------------------------------------------
    # 2. Recursive fallback
    # --------------------------------------------------------
    recursive_matches = []

    for root in unique_roots:

        print(f"Searching under: {root}")

        try:
            for fire_dir in root.rglob("fire_firms"):

                if not fire_dir.is_dir():
                    continue

                hourly = fire_dir / "hourly"
                daily = fire_dir / "daily"

                if has_firms_files(hourly):
                    recursive_matches.append(
                        (hourly, "hourly")
                    )

                elif has_firms_files(daily):
                    recursive_matches.append(
                        (daily, "daily")
                    )

        except (PermissionError, OSError):
            continue

    if recursive_matches:

        # Prefer hourly
        recursive_matches = sorted(
            recursive_matches,
            key=lambda x: (
                0 if x[1] == "hourly" else 1,
                len(str(x[0])),
            ),
        )

        return recursive_matches[0]

    return None, None


FIRE_FIRMS_DIR, FIRMS_SOURCE_LEVEL = find_firms_folder(
    DATASET_ROOT
)

if FIRE_FIRMS_DIR is None:

    raise FileNotFoundError(
        "\nRaw FIRMS files were not found.\n\n"
        "Notebook 16 needs the original KARHUT_Data_Share files, "
        "not only the processed skripsi outputs.\n\n"
        "Expected structure:\n"
        "KARHUT_Data_Share/\n"
        "└── data/\n"
        "    └── kalimantan/\n"
        "        └── fire_firms/\n"
        "            └── hourly/\n"
        "                ├── fires_2026-05-01.csv\n"
        "                ├── fires_2026-05-02.csv\n"
        "                └── ...\n\n"
        "If the dataset exists elsewhere, set:\n"
        'DATASET_ROOT = Path(r"C:\\path\\to\\KARHUT_Data_Share")'
    )


fire_files = sorted(
    FIRE_FIRMS_DIR.glob(
        "fires_*.csv"
    )
)

print()
print("Raw FIRMS folder :", FIRE_FIRMS_DIR)
print("Source level     :", FIRMS_SOURCE_LEVEL)
print("Daily files      :", len(fire_files))
print(
    "First / last     :",
    fire_files[0].name,
    "/",
    fire_files[-1].name,
)

Searching under: c:\Users\Asus\Documents\skripsi-karhut
Searching under: c:\Users\Asus\Documents
Searching under: C:\Users\Asus\Downloads

Raw FIRMS folder : c:\Users\Asus\Documents\skripsi-karhut\data\raw\sumatra\fire_firms\hourly
Source level     : hourly
Daily files      : 134
First / last     : fires_2026-05-01.csv / fires_2026-09-11.csv


## 4. Load raw FIRMS detections through the last complete analysis day

The previous pipeline treats 2026-09-11 as partial, so regional selection is based on **2026-05-01 through 2026-09-10**.

In [64]:
# 4. Load raw FIRMS detections

ANALYSIS_START = pd.Timestamp("2026-05-01")
ANALYSIS_END = pd.Timestamp("2026-09-10")

fire_frames = []

date_pattern = re.compile(r"fires_(\d{4}-\d{2}-\d{2})\.csv$")

for path in fire_files:
    match = date_pattern.search(path.name)

    if not match:
        continue

    date = pd.Timestamp(match.group(1))

    if not (ANALYSIS_START <= date <= ANALYSIS_END):
        continue

    day = pd.read_csv(path)

    if day.empty:
        continue

    rename_map = {}

    if "latitude" in day.columns and "lat" not in day.columns:
        rename_map["latitude"] = "lat"

    if "longitude" in day.columns and "lon" not in day.columns:
        rename_map["longitude"] = "lon"

    day = day.rename(columns=rename_map)

    required = {"lat", "lon"}

    if not required.issubset(day.columns):
        raise ValueError(
            f"{path.name} is missing {sorted(required - set(day.columns))}"
        )

    if "frp" not in day.columns:
        day["frp"] = np.nan

    day = day[["lat", "lon", "frp"]].copy()
    day["date"] = date

    fire_frames.append(day)

if not fire_frames:
    raise ValueError("No FIRMS rows loaded for the analysis period.")

fires_raw = pd.concat(
    fire_frames,
    ignore_index=True,
)

fires_raw["lat"] = pd.to_numeric(
    fires_raw["lat"],
    errors="coerce",
)

fires_raw["lon"] = pd.to_numeric(
    fires_raw["lon"],
    errors="coerce",
)

fires_raw["frp"] = pd.to_numeric(
    fires_raw["frp"],
    errors="coerce",
)

fires_raw = fires_raw.dropna(
    subset=["lat", "lon"]
).reset_index(drop=True)

print("Raw detections loaded :", len(fires_raw))
print("Date range            :", fires_raw["date"].min().date(), "→", fires_raw["date"].max().date())
print("Unique active days    :", fires_raw["date"].nunique())
display(fires_raw.head())

Raw detections loaded : 32191
Date range            : 2026-05-01 → 2026-09-10
Unique active days    : 124


,lat,lon,frp,date
0,-3.68855,103.65911,6.73,2026-05-01
1,-2.48685,105.17832,2.67,2026-05-01
2,-2.20366,103.00594,2.74,2026-05-01
3,1.67005,101.47509,2.37,2026-05-01
4,5.59902,102.74738,6.29,2026-05-01


# PART B — Natural Earth geographic context

## 5. Download / cache Natural Earth 1:10m layers

Layers:

- land;
- admin-0 countries;
- admin-1 states/provinces;
- lakes;
- rivers + lake centerlines.

The download is cached under:

```text
data/external/natural_earth/
```

so future runs do not redownload the files.

In [65]:
# 5. Natural Earth download helpers

NATURAL_EARTH = {
    "land": (
        "https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_land.zip",
        "ne_10m_land",
    ),
    "countries": (
        "https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_countries.zip",
        "ne_10m_admin_0_countries",
    ),
    "admin1": (
        "https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_1_states_provinces.zip",
        "ne_10m_admin_1_states_provinces",
    ),
    "lakes": (
        "https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_lakes.zip",
        "ne_10m_lakes",
    ),
    "rivers": (
        "https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_rivers_lake_centerlines.zip",
        "ne_10m_rivers_lake_centerlines",
    ),
}


def download_and_extract_ne(
    key,
):
    url, stem = NATURAL_EARTH[key]

    target_dir = (
        EXTERNAL_DIR
        / stem
    )

    shp_path = (
        target_dir
        / f"{stem}.shp"
    )

    if shp_path.exists():
        return shp_path

    target_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    zip_path = (
        EXTERNAL_DIR
        / f"{stem}.zip"
    )

    if not zip_path.exists():
        print("Downloading:", url)

        response = requests.get(
            url,
            timeout=120,
        )

        response.raise_for_status()

        zip_path.write_bytes(
            response.content
        )

    with zipfile.ZipFile(
        zip_path,
        "r",
    ) as archive:
        archive.extractall(
            target_dir
        )

    if not shp_path.exists():
        alternatives = list(
            target_dir.glob("*.shp")
        )

        if not alternatives:
            raise FileNotFoundError(
                f"No shapefile found after extracting {zip_path.name}"
            )

        shp_path = alternatives[0]

    return shp_path


ne_paths = {
    key: download_and_extract_ne(key)
    for key in NATURAL_EARTH
}

for key, path in ne_paths.items():
    print(f"{key:10s} -> {path}")

land       -> c:\Users\Asus\Documents\skripsi-karhut\data\external\natural_earth\ne_10m_land\ne_10m_land.shp
countries  -> c:\Users\Asus\Documents\skripsi-karhut\data\external\natural_earth\ne_10m_admin_0_countries\ne_10m_admin_0_countries.shp
admin1     -> c:\Users\Asus\Documents\skripsi-karhut\data\external\natural_earth\ne_10m_admin_1_states_provinces\ne_10m_admin_1_states_provinces.shp
lakes      -> c:\Users\Asus\Documents\skripsi-karhut\data\external\natural_earth\ne_10m_lakes\ne_10m_lakes.shp
rivers     -> c:\Users\Asus\Documents\skripsi-karhut\data\external\natural_earth\ne_10m_rivers_lake_centerlines\ne_10m_rivers_lake_centerlines.shp


## 6. Load layers and reconstruct Indonesian Kalimantan

As in the corrected spatial analysis:

1. select the main Borneo land polygon using a seed point around `(114°E, 0.5°N)`;
2. intersect it with Indonesia;
3. use that result as the valid Indonesian Kalimantan land geometry.

In [66]:
# 6. Build Indonesian Kalimantan land geometry

land = gpd.read_file(
    ne_paths["land"]
).to_crs(4326)

countries = gpd.read_file(
    ne_paths["countries"]
).to_crs(4326)

admin1 = gpd.read_file(
    ne_paths["admin1"]
).to_crs(4326)

lakes = gpd.read_file(
    ne_paths["lakes"]
).to_crs(4326)

rivers = gpd.read_file(
    ne_paths["rivers"]
).to_crs(4326)

land_parts = (
    land[["geometry"]]
    .explode(
        index_parts=False
    )
    .reset_index(drop=True)
)

borneo_seed = Point(
    114.0,
    0.5,
)

borneo_candidates = land_parts[
    land_parts.geometry.contains(
        borneo_seed
    )
]

if borneo_candidates.empty:
    raise RuntimeError(
        "Could not identify the main Borneo polygon from Natural Earth."
    )

borneo_geom = (
    borneo_candidates
    .iloc[0]
    .geometry
)

country_mask = pd.Series(
    False,
    index=countries.index,
)

if "ADM0_A3" in countries.columns:
    country_mask |= countries["ADM0_A3"].eq("IDN")

if "SOV_A3" in countries.columns:
    country_mask |= countries["SOV_A3"].eq("IDN")

if "ISO_A2" in countries.columns:
    country_mask |= countries["ISO_A2"].eq("ID")

if "ADMIN" in countries.columns:
    country_mask |= countries["ADMIN"].astype(str).str.lower().eq("indonesia")

indonesia = countries[
    country_mask
].copy()

if indonesia.empty:
    raise RuntimeError(
        "Could not identify Indonesia in Natural Earth admin-0."
    )

indonesia_geom = (
    indonesia.geometry
    .union_all()
)

kalimantan_geom = (
    indonesia_geom
    .intersection(
        borneo_geom
    )
)

kalimantan_gdf = gpd.GeoDataFrame(
    {
        "name": [
            "Indonesian Kalimantan"
        ]
    },
    geometry=[
        kalimantan_geom
    ],
    crs=4326,
)

print(
    "Kalimantan bounds:",
    tuple(
        round(v, 4)
        for v in kalimantan_geom.bounds
    ),
)

Kalimantan bounds: (108.8555, -4.1842, 119.0117, 4.3968)


## 7. Identify Kalimantan admin-1 provinces

Province labels are used only for geographic interpretation of candidate windows.

In [67]:
# 7. Indonesian admin-1 features intersecting Borneo

admin1_indonesia_mask = pd.Series(
    False,
    index=admin1.index,
)

for field, value in [
    ("adm0_a3", "IDN"),
    ("ADM0_A3", "IDN"),
    ("iso_a2", "ID"),
    ("ISO_A2", "ID"),
]:

    if field in admin1.columns:
        admin1_indonesia_mask |= (
            admin1[field]
            .astype(str)
            .eq(value)
        )

if "admin" in admin1.columns:
    admin1_indonesia_mask |= (
        admin1["admin"]
        .astype(str)
        .str.lower()
        .eq("indonesia")
    )

admin1_id = admin1[
    admin1_indonesia_mask
].copy()

admin1_kalimantan = admin1_id[
    admin1_id.geometry.intersects(
        borneo_geom
    )
].copy()

province_name_field = None

for candidate in [
    "name",
    "name_en",
    "gn_name",
    "woe_name",
]:

    if candidate in admin1_kalimantan.columns:
        province_name_field = candidate
        break

if province_name_field is None:
    province_name_field = admin1_kalimantan.columns[0]

province_names = (
    admin1_kalimantan[
        province_name_field
    ]
    .astype(str)
    .tolist()
)

print(
    "Admin-1 regions intersecting Indonesian Borneo:"
)

for name in province_names:
    print("-", name)

Admin-1 regions intersecting Indonesian Borneo:
- Kalimantan Timur
- Kalimantan Barat
- Kalimantan Tengah
- Kalimantan Selatan


## 8. Filter raw FIRMS detections to corrected Indonesian Kalimantan geometry

This removes detections outside the corrected land domain before regional selection.

In [68]:
# 8. Geometry filter for raw FIRMS

fire_points = gpd.GeoDataFrame(
    fires_raw.copy(),
    geometry=gpd.points_from_xy(
        fires_raw["lon"],
        fires_raw["lat"],
    ),
    crs=4326,
)

inside_mask = fire_points.geometry.within(
    kalimantan_geom
)

fires = (
    fire_points.loc[
        inside_mask,
        [
            "date",
            "lat",
            "lon",
            "frp",
            "geometry",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

print("Raw FIRMS rows       :", len(fires_raw))
print("Inside corrected land:", len(fires))
print(
    "Retained %           :",
    round(
        100
        * len(fires)
        / len(fires_raw),
        2,
    ),
)

Raw FIRMS rows       : 32191
Inside corrected land: 0
Retained %           : 0.0


In [69]:
# Load KALIMANTAN FIRMS explicitly
# Prevent accidental Sumatra source selection

from pathlib import Path
import re
import pandas as pd
import numpy as np
import geopandas as gpd

FIRE_FIRMS_DIR = Path(
    r"C:\Users\Asus\Documents\skripsi-karhut\data\raw\kalimantan\fire_firms\hourly"
)

if not FIRE_FIRMS_DIR.exists():
    raise FileNotFoundError(
        f"Kalimantan FIRMS folder not found:\n{FIRE_FIRMS_DIR}"
    )

if "kalimantan" not in str(FIRE_FIRMS_DIR).lower():
    raise RuntimeError(
        "Safety check failed: FIRMS path is not Kalimantan."
    )

fire_files = sorted(
    FIRE_FIRMS_DIR.glob("fires_*.csv")
)

if not fire_files:
    raise FileNotFoundError(
        f"No fires_*.csv files found in:\n{FIRE_FIRMS_DIR}"
    )

DATA_START = pd.Timestamp("2026-05-01")
DATA_END = pd.Timestamp("2026-09-10")

pattern = re.compile(
    r"fires_(\d{4}-\d{2}-\d{2})\.csv$"
)

frames = []

for path in fire_files:

    match = pattern.search(path.name)

    if not match:
        continue

    date = pd.Timestamp(
        match.group(1)
    )

    if not (
        DATA_START
        <= date
        <= DATA_END
    ):
        continue

    day = pd.read_csv(path)

    # Empty file = valid zero-detection day
    if day.empty:
        continue

    rename_map = {}

    if (
        "latitude" in day.columns
        and "lat" not in day.columns
    ):
        rename_map["latitude"] = "lat"

    if (
        "longitude" in day.columns
        and "lon" not in day.columns
    ):
        rename_map["longitude"] = "lon"

    day = day.rename(
        columns=rename_map
    )

    if not {
        "lat",
        "lon",
    }.issubset(day.columns):

        raise ValueError(
            f"Missing lat/lon in {path.name}"
        )

    if "frp" not in day.columns:
        day["frp"] = np.nan

    day = day[
        [
            "lat",
            "lon",
            "frp",
        ]
    ].copy()

    day["date"] = date

    frames.append(day)


if not frames:
    raise ValueError(
        "No Kalimantan FIRMS rows loaded."
    )


fires_raw = pd.concat(
    frames,
    ignore_index=True
)

fires_raw["lat"] = pd.to_numeric(
    fires_raw["lat"],
    errors="coerce"
)

fires_raw["lon"] = pd.to_numeric(
    fires_raw["lon"],
    errors="coerce"
)

fires_raw["frp"] = pd.to_numeric(
    fires_raw["frp"],
    errors="coerce"
)

fires_raw = (
    fires_raw
    .dropna(
        subset=[
            "lat",
            "lon",
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# Strong regional-source validation
# ============================================================

lon_min = float(
    fires_raw["lon"].min()
)

lon_max = float(
    fires_raw["lon"].max()
)

lat_min = float(
    fires_raw["lat"].min()
)

lat_max = float(
    fires_raw["lat"].max()
)


print("FIRMS source:")
print(FIRE_FIRMS_DIR)

print()
print(
    "Raw FIRMS rows:",
    len(fires_raw)
)

print(
    "Longitude:",
    round(lon_min, 4),
    "→",
    round(lon_max, 4)
)

print(
    "Latitude :",
    round(lat_min, 4),
    "→",
    round(lat_max, 4)
)


# Sumatra dataset would end around 106.5 E.
if lon_max < 108.0:

    raise ValueError(
        "WRONG FIRMS SOURCE: coordinates look like Sumatra, "
        "not Kalimantan."
    )


print()
print(
    "✅ Kalimantan FIRMS source check passed."
)

FIRMS source:
C:\Users\Asus\Documents\skripsi-karhut\data\raw\kalimantan\fire_firms\hourly

Raw FIRMS rows: 141628
Longitude: 108.0004 → 119.4999
Latitude : -4.2621 → 7.3858

✅ Kalimantan FIRMS source check passed.


In [70]:
# Filter FIRMS points to Indonesian Kalimantan geometry

if "kalimantan_geom" not in globals():
    raise RuntimeError(
        "kalimantan_geom is not defined. "
        "Rerun the Indonesian Kalimantan geometry cell."
    )


print(
    "FIRMS bounds:"
)

print(
    (
        float(fires_raw["lon"].min()),
        float(fires_raw["lat"].min()),
        float(fires_raw["lon"].max()),
        float(fires_raw["lat"].max()),
    )
)

print(
    "Kalimantan bounds:",
    kalimantan_geom.bounds
)


# ------------------------------------------------------------
# First check whether the bounding boxes overlap at all
# ------------------------------------------------------------

fire_minx = float(
    fires_raw["lon"].min()
)

fire_maxx = float(
    fires_raw["lon"].max()
)

fire_miny = float(
    fires_raw["lat"].min()
)

fire_maxy = float(
    fires_raw["lat"].max()
)

kal_minx, kal_miny, kal_maxx, kal_maxy = (
    kalimantan_geom.bounds
)

bbox_overlap = not (
    fire_maxx < kal_minx
    or fire_minx > kal_maxx
    or fire_maxy < kal_miny
    or fire_miny > kal_maxy
)

if not bbox_overlap:

    raise ValueError(
        "FIRMS bounds do not overlap Kalimantan geometry. "
        "Wrong regional source is still being used."
    )


# ------------------------------------------------------------
# Build point geometry
# ------------------------------------------------------------

fire_points = gpd.GeoDataFrame(
    fires_raw.copy(),
    geometry=gpd.points_from_xy(
        fires_raw["lon"],
        fires_raw["lat"],
    ),
    crs="EPSG:4326",
)


inside_mask = (
    fire_points.geometry
    .intersects(
        kalimantan_geom
    )
)


fires = (
    fire_points.loc[
        inside_mask,
        [
            "date",
            "lat",
            "lon",
            "frp",
            "geometry",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


print()
print(
    "Raw FIRMS rows       :",
    len(fires_raw)
)

print(
    "Inside Kalimantan    :",
    len(fires)
)

print(
    "Retention %          :",
    round(
        100
        * len(fires)
        / len(fires_raw),
        2
    )
)


if fires.empty:

    raise ValueError(
        "fires is still empty after Kalimantan geometry filtering."
    )


print()
print(
    "✅ Kalimantan geometry filtering passed."
)

FIRMS bounds:
(108.00037, -4.26212, 119.49989, 7.38579)
Kalimantan bounds: (108.85547936300003, -4.184177341999941, 119.01172936300009, 4.396787008000018)

Raw FIRMS rows       : 141628
Inside Kalimantan    : 135017
Retention %          : 95.33

✅ Kalimantan geometry filtering passed.


# PART C — Discover candidate regional hotspots

## 9. Build a 0.25° raw-detection density surface

This is only for **regional candidate discovery**.

It does not yet define the final modeling grid.

In [71]:
# 9. 0.25-degree density bins

DISCOVERY_STEP = 0.25
ANCHOR_LON = 108.0
ANCHOR_LAT = -4.5

if (
    "fires" not in globals()
    or fires is None
    or len(fires) == 0
):
    raise ValueError(
        "`fires` is empty before density binning."
    )

print(
    "Input Kalimantan FIRMS detections:",
    len(fires)
)

# Don't mutate original `fires`
discovery_fires = fires.copy()

discovery_fires["lon"] = pd.to_numeric(
    discovery_fires["lon"],
    errors="coerce"
)

discovery_fires["lat"] = pd.to_numeric(
    discovery_fires["lat"],
    errors="coerce"
)

discovery_fires = (
    discovery_fires
    .dropna(
        subset=["lon", "lat"]
    )
    .copy()
)

discovery_fires["disc_col"] = np.floor(
    (
        discovery_fires["lon"]
        - ANCHOR_LON
    )
    / DISCOVERY_STEP
).astype("int64")

discovery_fires["disc_row"] = np.floor(
    (
        discovery_fires["lat"]
        - ANCHOR_LAT
    )
    / DISCOVERY_STEP
).astype("int64")


density = (
    discovery_fires
    .groupby(
        [
            "disc_row",
            "disc_col",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "detections"
        }
    )
)


row_min = int(
    density["disc_row"].min()
)

row_max = int(
    density["disc_row"].max()
)

col_min = int(
    density["disc_col"].min()
)

col_max = int(
    density["disc_col"].max()
)


n_rows = (
    row_max
    - row_min
    + 1
)

n_cols = (
    col_max
    - col_min
    + 1
)


density_matrix = np.zeros(
    (
        n_rows,
        n_cols,
    ),
    dtype=np.float64,
)


for row in density.itertuples(
    index=False
):

    matrix_row = int(
        row.disc_row
        - row_min
    )

    matrix_col = int(
        row.disc_col
        - col_min
    )

    density_matrix[
        matrix_row,
        matrix_col,
    ] = float(
        row.detections
    )


print()
print(
    "Discovery density matrix:",
    density_matrix.shape
)

print(
    "Occupied 0.25° cells     :",
    len(density)
)

print(
    "Total detections          :",
    int(density_matrix.sum())
)

print(
    "Corrected FIRMS rows      :",
    len(discovery_fires)
)


assert (
    int(density_matrix.sum())
    == len(discovery_fires)
)

print(
    "\n✅ Density binning passed."
)

Input Kalimantan FIRMS detections: 135017

Discovery density matrix: (35, 41)
Occupied 0.25° cells     : 649
Total detections          : 135017
Corrected FIRMS rows      : 135017

✅ Density binning passed.


## 10. Smooth density using a local 5×5 neighborhood

At 0.25°, a 5×5 neighborhood represents roughly a 1.25° regional activity window.

No SciPy dependency is required.

In [72]:
# 10. Simple 5x5 neighborhood sum

def box_sum_2d(
    matrix,
    radius_cells=2,
):

    padded = np.pad(
        matrix,
        pad_width=radius_cells,
        mode="constant",
        constant_values=0,
    )

    output = np.zeros_like(
        matrix,
        dtype=float,
    )

    width = (
        2
        * radius_cells
        + 1
    )

    for dr in range(width):
        for dc in range(width):
            output += padded[
                dr:
                dr + matrix.shape[0],
                dc:
                dc + matrix.shape[1],
            ]

    return output


smoothed_density = box_sum_2d(
    density_matrix,
    radius_cells=2,
)

print(
    "Peak local 5x5 detections:",
    int(
        smoothed_density.max()
    ),
)

Peak local 5x5 detections: 20754


## 11. Select non-overlapping candidate centers

Candidate window size:

```text
3° × 3°
```

This is approximately a few hundred kilometers across and is large enough to contain a meaningful regional graph while still being much more specific than all Kalimantan.

Top peaks are forced to be geographically separated so we do not return several almost-identical boxes around one cluster.

In [73]:
# 3. Discover KALIMANTAN raw FIRMS folder
# Prefer hourly because it is the more complete record.

from pathlib import Path

# ============================================================
# OPTIONAL MANUAL OVERRIDE
# ============================================================
# Kalau tahu lokasi KARHUT_Data_Share, isi:
#
# DATASET_ROOT = Path(
#     r"C:\Users\Asus\Documents\KARHUT_Data_Share"
# )
#
# Kalau tidak tahu, biarkan None.
# ============================================================

DATASET_ROOT = None


def has_firms_files(folder):
    return (
        folder.exists()
        and folder.is_dir()
        and any(folder.glob("fires_*.csv"))
    )


def is_kalimantan_path(path):
    """
    Make sure the selected FIRMS folder belongs to Kalimantan,
    not Sumatra.
    """
    parts = [
        str(part).lower()
        for part in path.parts
    ]

    return "kalimantan" in parts


def find_kalimantan_firms_folder(dataset_root=None):

    search_roots = []

    if dataset_root is not None:
        search_roots.append(
            Path(dataset_root)
        )

    search_roots.extend([
        PROJECT_ROOT,
        PROJECT_ROOT.parent,
        Path.home() / "Documents",
        Path.home() / "Downloads",
    ])

    # Remove duplicate roots
    unique_roots = []
    seen = set()

    for root in search_roots:

        if not root.exists():
            continue

        try:
            resolved = root.resolve()
        except Exception:
            resolved = root

        key = str(resolved).lower()

        if key not in seen:
            unique_roots.append(root)
            seen.add(key)

    # --------------------------------------------------------
    # 1. Direct expected Kalimantan paths
    # --------------------------------------------------------

    direct_candidates = []

    for root in unique_roots:

        direct_candidates.extend([
            root
            / "data"
            / "kalimantan"
            / "fire_firms"
            / "hourly",

            root
            / "data"
            / "kalimantan"
            / "fire_firms"
            / "daily",

            root
            / "KARHUT_Data_Share"
            / "data"
            / "kalimantan"
            / "fire_firms"
            / "hourly",

            root
            / "KARHUT_Data_Share"
            / "data"
            / "kalimantan"
            / "fire_firms"
            / "daily",
        ])

    # Prefer hourly
    for candidate in direct_candidates:

        if (
            candidate.name.lower() == "hourly"
            and is_kalimantan_path(candidate)
            and has_firms_files(candidate)
        ):
            return candidate, "hourly"

    # Daily fallback
    for candidate in direct_candidates:

        if (
            candidate.name.lower() == "daily"
            and is_kalimantan_path(candidate)
            and has_firms_files(candidate)
        ):
            return candidate, "daily"

    # --------------------------------------------------------
    # 2. Recursive search, BUT Kalimantan only
    # --------------------------------------------------------

    matches = []

    for root in unique_roots:

        print(
            "Searching under:",
            root,
        )

        try:

            for fire_dir in root.rglob(
                "fire_firms"
            ):

                # CRITICAL:
                # skip Sumatra / other regions
                if not is_kalimantan_path(
                    fire_dir
                ):
                    continue

                hourly = (
                    fire_dir
                    / "hourly"
                )

                daily = (
                    fire_dir
                    / "daily"
                )

                if has_firms_files(
                    hourly
                ):
                    matches.append(
                        (
                            hourly,
                            "hourly",
                        )
                    )

                elif has_firms_files(
                    daily
                ):
                    matches.append(
                        (
                            daily,
                            "daily",
                        )
                    )

        except (
            PermissionError,
            OSError,
        ):
            continue

    if matches:

        # hourly first, then shortest path
        matches = sorted(
            matches,
            key=lambda item: (
                0
                if item[1] == "hourly"
                else 1,
                len(
                    str(
                        item[0]
                    )
                ),
            ),
        )

        return matches[0]

    return None, None


FIRE_FIRMS_DIR, FIRMS_SOURCE_LEVEL = (
    find_kalimantan_firms_folder(
        DATASET_ROOT
    )
)

if FIRE_FIRMS_DIR is None:

    raise FileNotFoundError(
        "Could not locate KALIMANTAN FIRMS files.\n\n"
        "Expected path similar to:\n"
        "KARHUT_Data_Share/data/kalimantan/"
        "fire_firms/hourly/fires_*.csv\n\n"
        "Set DATASET_ROOT manually if needed."
    )


fire_files = sorted(
    FIRE_FIRMS_DIR.glob(
        "fires_*.csv"
    )
)

print()
print(
    "Raw FIRMS folder :",
    FIRE_FIRMS_DIR,
)

print(
    "Source level     :",
    FIRMS_SOURCE_LEVEL,
)

print(
    "Files            :",
    len(
        fire_files
    ),
)

print(
    "First / last     :",
    fire_files[0].name,
    "/",
    fire_files[-1].name,
)

# Important validation
assert is_kalimantan_path(
    FIRE_FIRMS_DIR
), (
    "Selected FIRMS folder is NOT Kalimantan."
)

Searching under: c:\Users\Asus\Documents\skripsi-karhut
Searching under: c:\Users\Asus\Documents
Searching under: C:\Users\Asus\Downloads

Raw FIRMS folder : c:\Users\Asus\Documents\skripsi-karhut\data\raw\kalimantan\fire_firms\hourly
Source level     : hourly
Files            : 134
First / last     : fires_2026-05-01.csv / fires_2026-09-11.csv


# PART D — Fine-grid event diagnostic for candidate ranking

## 12. Helper: valid grid centers inside Indonesian Kalimantan

Candidate ranking uses a **0.25° preview grid**.

A grid cell is considered for the ranking if its center falls inside the corrected Indonesian Kalimantan land geometry.

Notebook 17 will perform a more rigorous overlap-based land/water mask.

In [74]:
# 12. Grid-center helper

def aligned_index_range(
    low,
    high,
    anchor,
    step,
):

    start = math.floor(
        (
            low
            - anchor
        )
        / step
    )

    end = math.floor(
        (
            high
            - anchor
        )
        / step
    )

    return range(
        start,
        end + 1,
    )


def valid_grid_centers(
    bbox_tuple,
    step,
):

    west, south, east, north = (
        bbox_tuple
    )

    rows = []

    for col_idx in aligned_index_range(
        west,
        east,
        ANCHOR_LON,
        step,
    ):

        cell_west = (
            ANCHOR_LON
            + col_idx
            * step
        )

        center_lon = (
            cell_west
            + step / 2
        )

        if not (
            west
            <= center_lon
            <= east
        ):
            continue

        for row_idx in aligned_index_range(
            south,
            north,
            ANCHOR_LAT,
            step,
        ):

            cell_south = (
                ANCHOR_LAT
                + row_idx
                * step
            )

            center_lat = (
                cell_south
                + step / 2
            )

            if not (
                south
                <= center_lat
                <= north
            ):
                continue

            center = Point(
                center_lon,
                center_lat,
            )

            if center.within(
                kalimantan_geom
            ):

                rows.append({
                    "row_idx": row_idx,
                    "col_idx": col_idx,
                    "center_lat": center_lat,
                    "center_lon": center_lon,
                    "cell_id": (
                        f"g{step:.2f}_"
                        f"{row_idx}_"
                        f"{col_idx}"
                    ),
                })

    return pd.DataFrame(
        rows
    )

## 13. Helper: build 3-day Emerging panel

Definition:

```text
Emerging at target day t+1
=
target cell active at t+1
AND
target cell had zero detected activity during t-2, t-1, t
```

As before, this means **emerging FIRMS detection**, not proven new fire ignition.

In [75]:
# 13. Candidate fine-grid panel statistics

ALL_DATES = pd.date_range(
    ANALYSIS_START,
    ANALYSIS_END,
    freq="D",
)


def candidate_event_stats(
    fire_df,
    bbox_tuple,
    step=0.25,
):

    west, south, east, north = (
        bbox_tuple
    )

    valid_cells = valid_grid_centers(
        bbox_tuple,
        step,
    )

    if valid_cells.empty:
        return {
            "grid_step": step,
            "valid_cells": 0,
            "active_cells": 0,
            "positive_cell_days": 0,
            "emerging3": 0,
            "persistent3": 0,
            "target_rows": 0,
            "target_positive_rate": np.nan,
        }

    regional = fire_df[
        fire_df["lon"].between(
            west,
            east,
            inclusive="both",
        )
        &
        fire_df["lat"].between(
            south,
            north,
            inclusive="both",
        )
    ].copy()

    regional["col_idx"] = np.floor(
        (
            regional["lon"]
            - ANCHOR_LON
        )
        / step
    ).astype(int)

    regional["row_idx"] = np.floor(
        (
            regional["lat"]
            - ANCHOR_LAT
        )
        / step
    ).astype(int)

    lookup = valid_cells[
        [
            "row_idx",
            "col_idx",
            "cell_id",
        ]
    ]

    regional = regional.merge(
        lookup,
        on=[
            "row_idx",
            "col_idx",
        ],
        how="inner",
    )

    active_daily = (
        regional
        .groupby(
            [
                "date",
                "cell_id",
            ],
            as_index=False,
        )
        .agg(
            fire_count=(
                "frp",
                "size",
            ),
            frp_sum=(
                "frp",
                "sum",
            ),
        )
    )

    full_index = pd.MultiIndex.from_product(
        [
            ALL_DATES,
            valid_cells[
                "cell_id"
            ],
        ],
        names=[
            "date",
            "cell_id",
        ],
    )

    panel = (
        active_daily
        .set_index(
            [
                "date",
                "cell_id",
            ]
        )
        .reindex(
            full_index
        )
        .reset_index()
    )

    panel[
        [
            "fire_count",
            "frp_sum",
        ]
    ] = (
        panel[
            [
                "fire_count",
                "frp_sum",
            ]
        ]
        .fillna(0)
    )

    panel["fire_active"] = (
        panel["fire_count"]
        > 0
    ).astype(int)

    panel = panel.sort_values(
        [
            "cell_id",
            "date",
        ]
    ).reset_index(drop=True)

    panel["recent3_incl_t"] = (
        panel
        .groupby(
            "cell_id"
        )[
            "fire_active"
        ]
        .transform(
            lambda s: (
                s.rolling(
                    3,
                    min_periods=3,
                )
                .sum()
            )
        )
    )

    panel["target_t1"] = (
        panel
        .groupby(
            "cell_id"
        )[
            "fire_active"
        ]
        .shift(-1)
    )

    model_panel = panel[
        panel["recent3_incl_t"].notna()
        &
        panel["target_t1"].notna()
    ].copy()

    model_panel["emerging3"] = (
        model_panel["target_t1"].eq(1)
        &
        model_panel["recent3_incl_t"].eq(0)
    )

    model_panel["persistent3"] = (
        model_panel["target_t1"].eq(1)
        &
        model_panel["recent3_incl_t"].gt(0)
    )

    return {
        "grid_step": step,
        "valid_cells": int(
            len(
                valid_cells
            )
        ),
        "active_cells": int(
            regional[
                "cell_id"
            ].nunique()
        ),
        "positive_cell_days": int(
            panel[
                "fire_active"
            ].sum()
        ),
        "emerging3": int(
            model_panel[
                "emerging3"
            ].sum()
        ),
        "persistent3": int(
            model_panel[
                "persistent3"
            ].sum()
        ),
        "target_rows": int(
            len(
                model_panel
            )
        ),
        "target_positive_rate": float(
            model_panel[
                "target_t1"
            ].mean()
        ),
    }

## 14. Compute candidate activity + 0.25° Emerging statistics

In [77]:
# 11. Candidate peak selection

N_CANDIDATES = 5
MIN_CENTER_SEPARATION_DEG = 2.0

WINDOW_WIDTH_DEG = 3.0
HALF_WIDTH = WINDOW_WIDTH_DEG / 2.0


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

required_vars = [
    "smoothed_density",
    "row_min",
    "col_min",
    "ANCHOR_LAT",
    "ANCHOR_LON",
    "DISCOVERY_STEP",
    "kalimantan_geom",
]

missing_vars = [
    var
    for var in required_vars
    if var not in globals()
]

if missing_vars:
    raise RuntimeError(
        "Missing prerequisite variables: "
        f"{missing_vars}. "
        "Rerun density + smoothing cells first."
    )


if smoothed_density.size == 0:
    raise ValueError(
        "smoothed_density is empty."
    )


# ------------------------------------------------------------
# Sort density peaks from highest to lowest
# ------------------------------------------------------------

flat_order = np.argsort(
    smoothed_density.ravel()
)[::-1]


candidate_centers = []


# ------------------------------------------------------------
# Select spatially separated candidate centers
# ------------------------------------------------------------

for flat_index in flat_order:

    matrix_row, matrix_col = (
        np.unravel_index(
            flat_index,
            smoothed_density.shape,
        )
    )

    disc_row = (
        int(matrix_row)
        + int(row_min)
    )

    disc_col = (
        int(matrix_col)
        + int(col_min)
    )


    center_lat = (
        ANCHOR_LAT
        + (
            disc_row
            + 0.5
        )
        * DISCOVERY_STEP
    )

    center_lon = (
        ANCHOR_LON
        + (
            disc_col
            + 0.5
        )
        * DISCOVERY_STEP
    )


    center_point = Point(
        center_lon,
        center_lat,
    )


    # Candidate center must be inside Indonesian Kalimantan
    if not center_point.within(
        kalimantan_geom
    ):
        continue


    # --------------------------------------------------------
    # Keep candidate centers spatially separated
    # --------------------------------------------------------

    too_close = False

    for previous in candidate_centers:

        distance_deg = math.hypot(
            center_lon
            - previous["center_lon"],

            center_lat
            - previous["center_lat"],
        )

        if (
            distance_deg
            < MIN_CENTER_SEPARATION_DEG
        ):
            too_close = True
            break


    if too_close:
        continue


    candidate_centers.append({
        "center_lon":
            float(center_lon),

        "center_lat":
            float(center_lat),

        "peak_density_score":
            float(
                smoothed_density[
                    matrix_row,
                    matrix_col,
                ]
            ),
    })


    if (
        len(candidate_centers)
        >= N_CANDIDATES
    ):
        break


# ------------------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------------------

if not candidate_centers:
    raise ValueError(
        "No candidate regional centers were found."
    )


candidates = pd.DataFrame(
    candidate_centers
)


candidates[
    "candidate_id"
] = [
    f"C{i + 1}"
    for i in range(
        len(candidates)
    )
]


# ------------------------------------------------------------
# 3 x 3 degree candidate bounding boxes
# ------------------------------------------------------------

candidates[
    "west"
] = (
    candidates["center_lon"]
    - HALF_WIDTH
)

candidates[
    "east"
] = (
    candidates["center_lon"]
    + HALF_WIDTH
)

candidates[
    "south"
] = (
    candidates["center_lat"]
    - HALF_WIDTH
)

candidates[
    "north"
] = (
    candidates["center_lat"]
    + HALF_WIDTH
)


print(
    "Candidate regions:",
    len(candidates)
)

display(
    candidates[
        [
            "candidate_id",
            "center_lon",
            "center_lat",
            "peak_density_score",
            "west",
            "south",
            "east",
            "north",
        ]
    ]
)

Candidate regions: 5


,candidate_id,center_lon,center_lat,peak_density_score,west,south,east,north
0,C1,113.875,-1.875,20754.0,112.375,-3.375,115.375,-0.375
1,C2,110.625,-1.875,20648.0,109.125,-3.375,112.125,-0.375
2,C3,109.875,0.125,9528.0,108.375,-1.375,111.375,1.625
3,C4,115.625,-2.875,6672.0,114.125,-4.375,117.125,-1.375
4,C5,117.375,2.375,5484.0,115.875,0.875,118.875,3.875


# PART E — Add province and hydrography context

## 15. Clip hydrography around Kalimantan once

In [ ]:
# 15. Broad Kalimantan hydrography clip

minx, miny, maxx, maxy = (
    kalimantan_geom.bounds
)

padding = 0.5

broad_bbox = box(
    minx - padding,
    miny - padding,
    maxx + padding,
    maxy + padding,
)

lakes_kalimantan = lakes[
    lakes.geometry.intersects(
        broad_bbox
    )
].copy()

rivers_kalimantan = rivers[
    rivers.geometry.intersects(
        broad_bbox
    )
].copy()

print("Kalimantan-context lakes :", len(lakes_kalimantan))
print("Kalimantan-context rivers:", len(rivers_kalimantan))

## 16. Candidate province names, lake area, and river length

Hydrography metrics are descriptive only.

They are **not included in candidate ranking**.

In [ ]:
# 16. Candidate hydrography context

candidate_context_rows = []

for row in candidate_stats.itertuples(
    index=False
):

    bbox_geom = box(
        row.west,
        row.south,
        row.east,
        row.north,
    )

    province_hits = (
        admin1_kalimantan[
            admin1_kalimantan.geometry.intersects(
                bbox_geom
            )
        ][
            province_name_field
        ]
        .astype(str)
        .tolist()
    )

    candidate_lakes = lakes_kalimantan[
        lakes_kalimantan.geometry.intersects(
            bbox_geom
        )
    ].copy()

    candidate_rivers = rivers_kalimantan[
        rivers_kalimantan.geometry.intersects(
            bbox_geom
        )
    ].copy()

    if candidate_lakes.empty:
        lake_area_km2 = 0.0
    else:
        lake_clipped = candidate_lakes.copy()
        lake_clipped["geometry"] = (
            lake_clipped.geometry.intersection(
                bbox_geom
            )
        )

        lake_area_km2 = float(
            lake_clipped
            .to_crs(6933)
            .geometry
            .area
            .sum()
            / 1e6
        )

    if candidate_rivers.empty:
        river_length_km = 0.0
    else:
        river_clipped = candidate_rivers.copy()
        river_clipped["geometry"] = (
            river_clipped.geometry.intersection(
                bbox_geom
            )
        )

        river_length_km = float(
            river_clipped
            .to_crs(32650)
            .geometry
            .length
            .sum()
            / 1000
        )

    candidate_context_rows.append({
        "candidate_id": row.candidate_id,
        "provinces": " | ".join(
            sorted(
                set(
                    province_hits
                )
            )
        ),
        "natural_earth_lakes_n": int(
            len(
                candidate_lakes
            )
        ),
        "natural_earth_lake_area_km2": lake_area_km2,
        "natural_earth_river_features_n": int(
            len(
                candidate_rivers
            )
        ),
        "natural_earth_river_length_km": river_length_km,
    })


candidate_context = pd.DataFrame(
    candidate_context_rows
)

candidate_table = candidate_stats.merge(
    candidate_context,
    on="candidate_id",
    how="left",
    validate="one_to_one",
)

display(
    candidate_table
)

# PART F — Rank candidates

## 17. Ranking rule

Primary sorting:

1. more 3-day Emerging positives at 0.25°;
2. more active days;
3. more active 0.25° cells;
4. more raw detections.

This intentionally **does not use model performance or hydrography**.

In [ ]:
# 17. Data-driven ranking

ranked_candidates = (
    candidate_table
    .sort_values(
        [
            "grid025_emerging3",
            "active_days",
            "grid025_active_cells",
            "detections",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

ranked_candidates[
    "selection_rank"
] = (
    np.arange(
        len(
            ranked_candidates
        )
    )
    + 1
)

ranked_candidates = ranked_candidates[
    [
        "selection_rank",
        *[
            c
            for c in ranked_candidates.columns
            if c != "selection_rank"
        ],
    ]
]

display(
    ranked_candidates
)

## 18. Provisional recommended study area

This is a **data-driven recommendation for the next experiment**, not an irreversible thesis decision.

You should still inspect the map before locking the study area.

In [ ]:
# 18. Recommended candidate

chosen = (
    ranked_candidates
    .iloc[0]
    .copy()
)

CHOSEN_ID = str(
    chosen[
        "candidate_id"
    ]
)

CHOSEN_BBOX = (
    float(
        chosen["west"]
    ),
    float(
        chosen["south"]
    ),
    float(
        chosen["east"]
    ),
    float(
        chosen["north"]
    ),
)

print("Recommended candidate :", CHOSEN_ID)
print("BBox                  :", CHOSEN_BBOX)
print("Provinces             :", chosen["provinces"])
print("Raw detections        :", int(chosen["detections"]))
print("Active days           :", int(chosen["active_days"]))
print("0.25° Emerging3       :", int(chosen["grid025_emerging3"]))
print("0.25° valid cells     :", int(chosen["grid025_valid_cells"]))

# PART G — Visualize all candidates on Kalimantan

## 19. Whole-island context map

In [ ]:
# 19. Kalimantan candidate overview

fig, ax = plt.subplots(
    figsize=(9, 9)
)

kalimantan_gdf.boundary.plot(
    ax=ax,
    linewidth=1.3,
)

admin1_kalimantan.boundary.plot(
    ax=ax,
    linewidth=0.5,
    alpha=0.6,
)

sample_n = min(
    25000,
    len(
        fires
    ),
)

fires.sample(
    n=sample_n,
    random_state=42,
).plot(
    ax=ax,
    markersize=2,
    alpha=0.18,
)

for row in ranked_candidates.itertuples(
    index=False
):

    rect = plt.Rectangle(
        (
            row.west,
            row.south,
        ),
        row.east - row.west,
        row.north - row.south,
        fill=False,
        linewidth=2.0
        if row.selection_rank == 1
        else 1.0,
    )

    ax.add_patch(
        rect
    )

    ax.text(
        row.west + 0.05,
        row.north - 0.15,
        (
            f"{row.candidate_id}"
            f" (rank {row.selection_rank})"
        ),
        fontsize=9,
    )

ax.set_xlabel(
    "Longitude"
)

ax.set_ylabel(
    "Latitude"
)

ax.set_title(
    "Data-driven regional candidates — Indonesian Kalimantan"
)

ax.set_aspect(
    "equal",
    adjustable="box",
)

plt.tight_layout()

overview_path = (
    FIGURES_DIR
    / "01_kalimantan_candidate_regions.png"
)

plt.savefig(
    overview_path,
    dpi=180,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    overview_path,
)

# PART H — Detailed view of the recommended region

## 20. Clip FIRMS, provinces, lakes, and rivers to chosen region

In [ ]:
# 20. Chosen regional layers

chosen_geom = box(
    *CHOSEN_BBOX
)

chosen_fires = fires[
    fires.geometry.within(
        chosen_geom
    )
].copy()

chosen_provinces = admin1_kalimantan[
    admin1_kalimantan.geometry.intersects(
        chosen_geom
    )
].copy()

chosen_lakes = lakes_kalimantan[
    lakes_kalimantan.geometry.intersects(
        chosen_geom
    )
].copy()

chosen_rivers = rivers_kalimantan[
    rivers_kalimantan.geometry.intersects(
        chosen_geom
    )
].copy()

print("Chosen FIRMS detections:", len(chosen_fires))
print("Province polygons      :", len(chosen_provinces))
print("Lake features          :", len(chosen_lakes))
print("River features         :", len(chosen_rivers))

## 21. Static regional hydrography map

Raw FIRMS detections are overlaid with province boundaries, lakes, and rivers.

Interpret this as **geographic context**, not proof that individual detections are correctly classified as land/water at sub-kilometer scale.

In [ ]:
# 21. Detailed static map

fig, ax = plt.subplots(
    figsize=(9, 9)
)

chosen_provinces.boundary.plot(
    ax=ax,
    linewidth=1.0,
    alpha=0.8,
)

if not chosen_lakes.empty:
    chosen_lakes.plot(
        ax=ax,
        alpha=0.45,
    )

if not chosen_rivers.empty:
    chosen_rivers.plot(
        ax=ax,
        linewidth=0.7,
        alpha=0.65,
    )

chosen_fires.plot(
    ax=ax,
    markersize=5,
    alpha=0.22,
)

ax.set_xlim(
    CHOSEN_BBOX[0],
    CHOSEN_BBOX[2],
)

ax.set_ylim(
    CHOSEN_BBOX[1],
    CHOSEN_BBOX[3],
)

ax.set_xlabel(
    "Longitude"
)

ax.set_ylabel(
    "Latitude"
)

ax.set_title(
    f"{CHOSEN_ID} — FIRMS + Natural Earth lakes/rivers"
)

ax.set_aspect(
    "equal",
    adjustable="box",
)

plt.tight_layout()

detail_path = (
    FIGURES_DIR
    / "02_chosen_region_hydrography.png"
)

plt.savefig(
    detail_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    detail_path,
)

## 22. Optional interactive map

If `folium` is installed, this creates an HTML map that can be zoomed interactively in the browser.

The FIRMS layer uses a heatmap to keep the file smaller than rendering tens of thousands of individual markers.

In [ ]:
# 22. Interactive regional map

interactive_map_path = (
    FIGURES_DIR
    / "03_chosen_region_interactive_map.html"
)

if FOLIUM_AVAILABLE:

    center_lat = (
        CHOSEN_BBOX[1]
        + CHOSEN_BBOX[3]
    ) / 2

    center_lon = (
        CHOSEN_BBOX[0]
        + CHOSEN_BBOX[2]
    ) / 2

    fmap = folium.Map(
        location=[
            center_lat,
            center_lon,
        ],
        zoom_start=8,
        tiles="OpenStreetMap",
        control_scale=True,
    )

    folium.GeoJson(
        chosen_provinces.to_json(),
        name="Province boundaries",
        style_function=lambda feature: {
            "fillOpacity": 0.0,
            "weight": 1.2,
        },
    ).add_to(
        fmap
    )

    if not chosen_lakes.empty:
        folium.GeoJson(
            chosen_lakes.to_json(),
            name="Natural Earth lakes",
            style_function=lambda feature: {
                "fillOpacity": 0.4,
                "weight": 0.6,
            },
        ).add_to(
            fmap
        )

    if not chosen_rivers.empty:
        folium.GeoJson(
            chosen_rivers.to_json(),
            name="Natural Earth rivers",
            style_function=lambda feature: {
                "weight": 1.1,
            },
        ).add_to(
            fmap
        )

    heat_points = (
        chosen_fires[
            [
                "lat",
                "lon",
            ]
        ]
        .dropna()
        .values
        .tolist()
    )

    if heat_points:
        HeatMap(
            heat_points,
            radius=7,
            blur=10,
            min_opacity=0.25,
            name="FIRMS detection density",
        ).add_to(
            fmap
        )

    folium.Rectangle(
        bounds=[
            [
                CHOSEN_BBOX[1],
                CHOSEN_BBOX[0],
            ],
            [
                CHOSEN_BBOX[3],
                CHOSEN_BBOX[2],
            ],
        ],
        fill=False,
        weight=2,
        tooltip=CHOSEN_ID,
    ).add_to(
        fmap
    )

    folium.LayerControl(
        collapsed=False
    ).add_to(
        fmap
    )

    fmap.save(
        interactive_map_path
    )

    print(
        "Saved:",
        interactive_map_path,
    )

else:

    print(
        "Folium is not installed. "
        "Install with `python -m pip install folium` to generate the interactive HTML map."
    )

# PART I — Build rigorous land/water preview grids

## 23. Fine-grid polygon helper

For every cell we calculate:

- cell center;
- `land_fraction` = overlap with corrected Indonesian Kalimantan land;
- `water_fraction = 1 - land_fraction`;
- `lake_fraction` = overlap with Natural Earth lake polygons;
- `river_intersects` = whether a mapped Natural Earth river crosses the cell.

`water_fraction` captures coastline/ocean and large non-land areas in the Natural Earth geometry. `lake_fraction` is reported separately.

For final thesis modeling, do not treat `river_intersects=True` as “this whole cell is water”; rivers are lines and most cells remain predominantly land.

In [ ]:
# 23. Land/water fine-grid builder

def build_landwater_grid(
    bbox_tuple,
    step,
    min_land_fraction=0.05,
):

    west, south, east, north = (
        bbox_tuple
    )

    rows = []

    for col_idx in aligned_index_range(
        west,
        east,
        ANCHOR_LON,
        step,
    ):

        cell_west = (
            ANCHOR_LON
            + col_idx
            * step
        )

        cell_east = (
            cell_west
            + step
        )

        if (
            cell_east <= west
            or cell_west >= east
        ):
            continue

        for row_idx in aligned_index_range(
            south,
            north,
            ANCHOR_LAT,
            step,
        ):

            cell_south = (
                ANCHOR_LAT
                + row_idx
                * step
            )

            cell_north = (
                cell_south
                + step
            )

            if (
                cell_north <= south
                or cell_south >= north
            ):
                continue

            geom = box(
                cell_west,
                cell_south,
                cell_east,
                cell_north,
            )

            rows.append({
                "row_idx": row_idx,
                "col_idx": col_idx,
                "cell_id": (
                    f"g{step:.2f}_"
                    f"{row_idx}_"
                    f"{col_idx}"
                ),
                "center_lon": (
                    cell_west
                    + step / 2
                ),
                "center_lat": (
                    cell_south
                    + step / 2
                ),
                "geometry": geom,
            })

    grid = gpd.GeoDataFrame(
        rows,
        geometry="geometry",
        crs=4326,
    )

    if grid.empty:
        return grid

    grid_equal = grid.to_crs(
        6933
    )

    kalimantan_equal = (
        kalimantan_gdf
        .to_crs(
            6933
        )
        .geometry
        .iloc[0]
    )

    grid["cell_area_m2"] = (
        grid_equal.geometry.area
        .to_numpy()
    )

    land_intersection_area = (
        grid_equal.geometry
        .intersection(
            kalimantan_equal
        )
        .area
        .to_numpy()
    )

    grid["land_fraction"] = (
        land_intersection_area
        / grid["cell_area_m2"]
    )

    grid["water_fraction"] = (
        1.0
        - grid["land_fraction"]
    ).clip(
        0.0,
        1.0,
    )

    if chosen_lakes.empty:

        grid["lake_fraction"] = 0.0

    else:

        lakes_union_equal = (
            chosen_lakes
            .to_crs(
                6933
            )
            .geometry
            .union_all()
        )

        lake_intersection_area = (
            grid_equal.geometry
            .intersection(
                lakes_union_equal
            )
            .area
            .to_numpy()
        )

        grid["lake_fraction"] = (
            lake_intersection_area
            / grid["cell_area_m2"]
        ).clip(
            0.0,
            1.0,
        )

    if chosen_rivers.empty:

        grid["river_intersects"] = False

    else:

        rivers_union = (
            chosen_rivers.geometry
            .union_all()
        )

        grid["river_intersects"] = (
            grid.geometry.intersects(
                rivers_union
            )
        )

    center_points = gpd.GeoSeries(
        gpd.points_from_xy(
            grid["center_lon"],
            grid["center_lat"],
        ),
        crs=4326,
    )

    grid["center_inside_land"] = (
        center_points.within(
            kalimantan_geom
        )
        .to_numpy()
    )

    grid["keep_candidate"] = (
        grid["land_fraction"].ge(
            min_land_fraction
        )
        |
        grid["center_inside_land"]
    )

    grid["land_dominant"] = (
        grid["land_fraction"]
        >= 0.50
    )

    grid["water_dominant"] = (
        grid["land_fraction"]
        < 0.50
    )

    grid["lake_dominant"] = (
        grid["lake_fraction"]
        >= 0.50
    )

    return grid

## 24. Compare 0.25° and 0.10° spatial realism

In [ ]:
# 24. Resolution grid summaries

resolution_grids = {}
resolution_summary_rows = []

for step in [
    0.25,
    0.10,
]:

    grid = build_landwater_grid(
        CHOSEN_BBOX,
        step,
        min_land_fraction=0.05,
    )

    resolution_grids[
        step
    ] = grid

    kept = grid[
        grid[
            "keep_candidate"
        ]
    ]

    resolution_summary_rows.append({
        "Grid Step": step,
        "All Rectangular Cells": len(
            grid
        ),
        "Candidate Cells After Land Filter": len(
            kept
        ),
        "Land-Dominant Cells": int(
            kept[
                "land_dominant"
            ].sum()
        ),
        "Water-Dominant Mixed Cells": int(
            kept[
                "water_dominant"
            ].sum()
        ),
        "Lake-Dominant Cells": int(
            kept[
                "lake_dominant"
            ].sum()
        ),
        "Cells Crossed by Mapped River": int(
            kept[
                "river_intersects"
            ].sum()
        ),
        "Median Land Fraction": float(
            kept[
                "land_fraction"
            ].median()
        ),
    })


resolution_geo_summary = pd.DataFrame(
    resolution_summary_rows
)

display(
    resolution_geo_summary
)

## 25. Compare event counts at 0.25° and 0.10°

This checks whether finer grids create a more informative Emerging diagnostic rather than simply increasing node count.

In [ ]:
# 25. Resolution event summary

resolution_event_rows = []

for step in [
    0.25,
    0.10,
]:

    stats = candidate_event_stats(
        fires,
        CHOSEN_BBOX,
        step=step,
    )

    resolution_event_rows.append({
        "Grid Step": step,
        **{
            key: value
            for key, value in stats.items()
            if key != "grid_step"
        },
    })


resolution_event_summary = pd.DataFrame(
    resolution_event_rows
)

display(
    resolution_event_summary
)

# PART J — Visualize fine-grid land/water structure

## 26. 0.25° grid preview

In [ ]:
# 26. 0.25-degree land/water grid

grid025 = resolution_grids[
    0.25
]

fig, ax = plt.subplots(
    figsize=(9, 9)
)

grid025.plot(
    ax=ax,
    column="land_fraction",
    legend=True,
    alpha=0.55,
    edgecolor="black",
    linewidth=0.35,
)

if not chosen_lakes.empty:
    chosen_lakes.boundary.plot(
        ax=ax,
        linewidth=1.0,
    )

if not chosen_rivers.empty:
    chosen_rivers.plot(
        ax=ax,
        linewidth=0.6,
        alpha=0.7,
    )

chosen_fires.sample(
    n=min(
        8000,
        len(
            chosen_fires
        ),
    ),
    random_state=42,
).plot(
    ax=ax,
    markersize=3,
    alpha=0.20,
)

ax.set_xlim(
    CHOSEN_BBOX[0],
    CHOSEN_BBOX[2],
)

ax.set_ylim(
    CHOSEN_BBOX[1],
    CHOSEN_BBOX[3],
)

ax.set_title(
    f"{CHOSEN_ID} — 0.25° grid, land fraction + hydrography"
)

ax.set_xlabel(
    "Longitude"
)

ax.set_ylabel(
    "Latitude"
)

ax.set_aspect(
    "equal",
    adjustable="box",
)

plt.tight_layout()

path025 = (
    FIGURES_DIR
    / "04_chosen_region_grid025_land_fraction.png"
)

plt.savefig(
    path025,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    path025,
)

## 27. 0.10° grid preview

In [ ]:
# 27. 0.10-degree land/water grid

grid010 = resolution_grids[
    0.10
]

fig, ax = plt.subplots(
    figsize=(9, 9)
)

grid010.plot(
    ax=ax,
    column="land_fraction",
    legend=True,
    alpha=0.55,
    edgecolor="black",
    linewidth=0.20,
)

if not chosen_lakes.empty:
    chosen_lakes.boundary.plot(
        ax=ax,
        linewidth=1.0,
    )

if not chosen_rivers.empty:
    chosen_rivers.plot(
        ax=ax,
        linewidth=0.55,
        alpha=0.65,
    )

chosen_fires.sample(
    n=min(
        8000,
        len(
            chosen_fires
        ),
    ),
    random_state=42,
).plot(
    ax=ax,
    markersize=3,
    alpha=0.18,
)

ax.set_xlim(
    CHOSEN_BBOX[0],
    CHOSEN_BBOX[2],
)

ax.set_ylim(
    CHOSEN_BBOX[1],
    CHOSEN_BBOX[3],
)

ax.set_title(
    f"{CHOSEN_ID} — 0.10° grid, land fraction + hydrography"
)

ax.set_xlabel(
    "Longitude"
)

ax.set_ylabel(
    "Latitude"
)

ax.set_aspect(
    "equal",
    adjustable="box",
)

plt.tight_layout()

path010 = (
    FIGURES_DIR
    / "05_chosen_region_grid010_land_fraction.png"
)

plt.savefig(
    path010,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    path010,
)

# PART K — Save outputs for Notebook 17

In [ ]:
# 28. Save candidate tables

ranked_candidates.to_csv(
    RESULTS_DIR
    / "regional_candidate_ranking.csv",
    index=False,
)

resolution_geo_summary.to_csv(
    RESULTS_DIR
    / "chosen_region_resolution_geography.csv",
    index=False,
)

resolution_event_summary.to_csv(
    RESULTS_DIR
    / "chosen_region_resolution_events.csv",
    index=False,
)

print(
    "Saved candidate / resolution summary tables."
)

In [ ]:
# 29. Save chosen-region raw FIRMS subset

chosen_fires.drop(
    columns=[
        "geometry"
    ]
).to_csv(
    EXPORT_DIR
    / "chosen_region_raw_firms.csv",
    index=False,
)

print(
    "Saved:",
    EXPORT_DIR
    / "chosen_region_raw_firms.csv",
)

In [ ]:
# 30. Save chosen regional geography

chosen_provinces.to_file(
    EXPORT_DIR
    / "chosen_region_provinces.geojson",
    driver="GeoJSON",
)

chosen_lakes.to_file(
    EXPORT_DIR
    / "chosen_region_lakes_naturalearth.geojson",
    driver="GeoJSON",
)

chosen_rivers.to_file(
    EXPORT_DIR
    / "chosen_region_rivers_naturalearth.geojson",
    driver="GeoJSON",
)

for step, grid in resolution_grids.items():

    safe_step = (
        str(step)
        .replace(
            ".",
            "p",
        )
    )

    grid.to_file(
        EXPORT_DIR
        / f"chosen_region_grid_{safe_step}.geojson",
        driver="GeoJSON",
    )

print(
    "Saved regional GeoJSON outputs to:",
    EXPORT_DIR,
)

In [ ]:
# 31. Save chosen-region metadata

chosen_metadata = {
    "candidate_id": CHOSEN_ID,
    "selection_rule": [
        "grid025_emerging3 descending",
        "active_days descending",
        "grid025_active_cells descending",
        "detections descending",
    ],
    "bbox_wgs84": {
        "west": CHOSEN_BBOX[0],
        "south": CHOSEN_BBOX[1],
        "east": CHOSEN_BBOX[2],
        "north": CHOSEN_BBOX[3],
    },
    "provinces": str(
        chosen[
            "provinces"
        ]
    ).split(
        " | "
    ),
    "analysis_period": {
        "start": str(
            ANALYSIS_START.date()
        ),
        "end": str(
            ANALYSIS_END.date()
        ),
    },
    "raw_detection_count": int(
        chosen[
            "detections"
        ]
    ),
    "active_days": int(
        chosen[
            "active_days"
        ]
    ),
    "grid025_emerging3": int(
        chosen[
            "grid025_emerging3"
        ]
    ),
    "hydrography_source": (
        "Natural Earth 1:10m; screening context only, not exhaustive final water mask"
    ),
}

metadata_path = (
    EXPORT_DIR
    / "chosen_region_metadata.json"
)

metadata_path.write_text(
    json.dumps(
        chosen_metadata,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Saved:",
    metadata_path,
)

# PART L — Automatic interpretation

## 32. Regional zoom decision summary

This cell does **not** automatically declare one fine-grid resolution “best”.

Instead it reports:

- chosen subregion;
- geographic context;
- 0.25° vs 0.10° node counts;
- Emerging-event counts;
- whether 0.10° appears to reveal more spatially localized events.

The final resolution decision should also consider weather-source resolution in Notebook 17.

In [ ]:
# 32. Automatic summary

event_map = (
    resolution_event_summary
    .set_index(
        "Grid Step"
    )
)

geo_map = (
    resolution_geo_summary
    .set_index(
        "Grid Step"
    )
)

emerging_025 = int(
    event_map.loc[
        0.25,
        "emerging3",
    ]
)

emerging_010 = int(
    event_map.loc[
        0.10,
        "emerging3",
    ]
)

cells_025 = int(
    geo_map.loc[
        0.25,
        "Candidate Cells After Land Filter",
    ]
)

cells_010 = int(
    geo_map.loc[
        0.10,
        "Candidate Cells After Land Filter",
    ]
)

print(
    "=== Notebook 16 regional zoom summary ==="
)

print()

print(
    "SELECTED REGION"
)

print(
    "Candidate             :",
    CHOSEN_ID,
)

print(
    "BBox WGS84            :",
    CHOSEN_BBOX,
)

print(
    "Province context      :",
    chosen[
        "provinces"
    ],
)

print(
    "Raw FIRMS detections  :",
    int(
        chosen[
            "detections"
        ]
    ),
)

print(
    "Active days           :",
    int(
        chosen[
            "active_days"
        ]
    ),
)

print()

print(
    "HYDROGRAPHY SCREENING"
)

print(
    "Mapped lake features  :",
    int(
        chosen[
            "natural_earth_lakes_n"
        ]
    ),
)

print(
    "Mapped river features :",
    int(
        chosen[
            "natural_earth_river_features_n"
        ]
    ),
)

print()

print(
    "FINE-GRID PREVIEW"
)

print(
    "0.25° candidate cells :",
    cells_025,
)

print(
    "0.25° Emerging3       :",
    emerging_025,
)

print(
    "0.10° candidate cells :",
    cells_010,
)

print(
    "0.10° Emerging3       :",
    emerging_010,
)

print()

if emerging_010 > emerging_025:

    print(
        "The 0.10° preview exposes more 3-day Emerging locations than 0.25°, "
        "which supports the hypothesis that the original coarse grid was hiding "
        "localized next-day hotspot emergence."
    )

else:

    print(
        "The 0.10° preview does not increase the Emerging-event count. "
        "Finer spatial resolution therefore should not be adopted solely on the "
        "assumption that it creates a more informative Emerging task."
    )

print()

print(
    "NEXT:"
)

print(
    "Use Notebook 17 to build the selected regional dataset. "
    "Compare 0.25° and 0.10° while explicitly checking weather-resolution mismatch "
    "and replacing the Natural Earth screening layer with a higher-resolution "
    "water mask if the thesis requires detailed lake/river exclusion."
)

# What to send for review

After running Notebook 16, send:

1. **Section 17** — ranked candidate table
2. **Section 18** — selected region summary
3. **Section 21** — detailed hydrography map
4. **Section 24** — 0.25° vs 0.10° geographic grid summary
5. **Section 25** — 0.25° vs 0.10° event summary
6. **Section 32** — automatic interpretation

Also open:

```text
model_results/regional_zoom_selection/figures/03_chosen_region_interactive_map.html
```

in your browser if Folium is available.

---

## Important interpretation language

Use:

> “Subregion dipilih berdasarkan kepadatan dan kecukupan aktivitas FIRMS, khususnya jumlah Emerging detection pada preview grid yang lebih halus.”

Avoid:

> “Ini daerah dengan kebakaran terbanyak sehingga pasti daerah terbaik untuk model.”

And for water:

> “Natural Earth digunakan sebagai hydrography screening layer. Final fine-grid water exclusion memerlukan mask beresolusi lebih tinggi.”

Avoid:

> “Semua sungai dan danau sudah terpetakan secara lengkap.”